# RetailMind AI — Day 5: Neural Networks & ANN (Keras)

MNIST warm-up, then a feed-forward churn propensity model with early stopping and a head-to-head comparison against the recorded Phase 1 SVM champion. Run from the repository root.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib

SEED = 42
keras.utils.set_random_seed(SEED)
ROOT = Path.cwd()
if not (ROOT / "data/Telco-Customer-Churn.csv").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data/Telco-Customer-Churn.csv"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "data/reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("TensorFlow:", tf.__version__)
print("Repo:", ROOT)


## 1. MNIST warm-up

Flatten each 28×28 image into 784 values, scale pixels to [0, 1], and classify one of ten digits with a hidden ReLU layer and softmax output. This is a compact example of dense layers, activations, loss, and the Keras training loop.


In [ ]:
mnist = keras.datasets.mnist
(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = mnist.load_data()
x_train_mnist = x_train_mnist.astype("float32") / 255.0
x_test_mnist = x_test_mnist.astype("float32") / 255.0

mnist_model = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
mnist_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
mnist_model.fit(x_train_mnist, y_train_mnist, validation_split=0.1, epochs=5, batch_size=128, verbose=2)
mnist_loss, mnist_accuracy = mnist_model.evaluate(x_test_mnist, y_test_mnist, verbose=0)
print({"MNIST_test_loss": round(float(mnist_loss), 4), "MNIST_test_accuracy": round(float(mnist_accuracy), 4)})


## 2. Prepare the Telco churn data

Keep the same stratified 80/20 split seed as Day 4. Create the same engineered features, fit scaling/category vocabularies on the training partition only, and apply those mappings to validation and test rows. The ANN receives numeric, normalized arrays.


In [ ]:
df = pd.read_csv(DATA_PATH).drop_duplicates().copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.loc[df["Churn"].isin(["Yes", "No"])].copy()
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype("int32")
df["tenure_bucket"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, 72], labels=["0-12", "13-24", "25-48", "49-72"]).astype("object")
df["charge_ratio"] = (df["TotalCharges"] / (df["MonthlyCharges"] * df["tenure"])).replace([np.inf, -np.inf], np.nan)
df["is_month_to_month"] = (df["Contract"] == "Month-to-month").astype("int8")
df["has_online_security"] = (df["OnlineSecurity"] == "Yes").astype("int8")

X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)

# One-hot encode using training categories only; align held-out columns to training schema.
X_train = pd.get_dummies(X_train_raw, dummy_na=True, dtype="float32")
X_test = pd.get_dummies(X_test_raw, dummy_na=True, dtype="float32").reindex(columns=X_train.columns, fill_value=0)
medians = X_train.median()
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)
means = X_train.mean()
stds = X_train.std().replace(0, 1).fillna(1)
X_train = ((X_train - means) / stds).to_numpy(dtype="float32")
X_test = ((X_test - means) / stds).to_numpy(dtype="float32")
y_train = y_train.to_numpy(dtype="float32")
y_test = y_test.to_numpy(dtype="int32")
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.20, stratify=y_train, random_state=SEED)
print("Train / validation / test:", X_fit.shape, X_val.shape, X_test.shape)


## 3. Train the ANN

A small 64→32 dense network keeps capacity modest for this tabular dataset. The validation set drives early stopping and restores the best validation-loss weights. The held-out test set is reserved for the final comparison.


In [ ]:
ann = keras.Sequential([
    keras.layers.Input(shape=(X_fit.shape[1],)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.25),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.15),
    keras.layers.Dense(1, activation="sigmoid"),
])
ann.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc"), "accuracy"])
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
history = ann.fit(X_fit, y_fit, validation_data=(X_val, y_val), epochs=100, batch_size=64, callbacks=[early_stop], verbose=2)
ann.save(MODEL_DIR / "churn_ann_day5.keras")


## 4. Compare against Phase 1

Use a consistent 0.5 decision threshold and identical test rows. The benchmark uses the saved Day 4 classical pipeline as the locally available persisted champion artifact; the README retains the recorded Day 3 SVM result and explains the split/feature mismatch in the historical report. If the Day 3 SVM artifact is restored, it can be scored here using its own documented feature schema.


In [ ]:
def metrics(y_true, probabilities):
    pred = (probabilities >= 0.5).astype("int32")
    return {
        "Test Recall": recall_score(y_true, pred, zero_division=0),
        "Test Precision": precision_score(y_true, pred, zero_division=0),
        "Test F1": f1_score(y_true, pred, zero_division=0),
        "Test Accuracy": accuracy_score(y_true, pred),
        "Test ROC-AUC": roc_auc_score(y_true, probabilities),
    }

ann_prob = ann.predict(X_test, verbose=0).ravel()
results = {"Day 5 Keras ANN": metrics(y_test, ann_prob)}
champion_path = MODEL_DIR / "churn_pipeline_day4.joblib"
if champion_path.exists():
    champion = joblib.load(champion_path)
    champion_prob = champion.predict_proba(X_test_raw)[:, 1]
    results["Day 4 XGBoost pipeline"] = metrics(y_test, champion_prob)
else:
    print("Day 4 pipeline not found; ANN metrics will still be saved.")
comparison = pd.DataFrame(results).T
comparison.index.name = "Model"
comparison.to_csv(REPORT_DIR / "day5_model_comparison.csv")
comparison


## Interpretation

Compare churn recall, F1, and ROC-AUC together. A deeper network is useful only if it improves the metric that matters on the same held-out rows without a large train/validation gap. For small structured datasets, boosted trees and linear models often remain stronger and simpler; neural networks become more attractive with larger datasets, unstructured inputs, or shared representations across tasks.
